# PolyWhisper: Fine-Tuning Whisper-Small on Hinglish Speech
This notebook fine-tunes **OpenAI Whisper-Small** using a parameter-efficient **LoRA (Low-Rank Adaptation)** adapter on the **MUCS-Hinglish** code-mixed speech dataset via the PolyWhisper framework.

---

### Step 1: Install Dependencies & Clone PolyWhisper Framework
- Installs required audio processing, evaluation, and machine learning libraries (	ransformers, datasets, ccelerate, soundfile, jiwer).
- Clones the PolyWhisper repository which contains the multi-lingual Whisper LoRA training and inference architecture.

In [1]:
!pip -q install "datasets<4" transformers accelerate soundfile librosa resampy tqdm jiwer huggingface_hub
!git clone -q https://github.com/eulogik/PolyWhisper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 76.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pathos 0.3.5 requires dill>=0.4.1, but you have dill 0.3.8 which is incompatible.
pathos 0.3.5 requires multiprocess>=0.70.19, but you have multiprocess 0.70.16 which is incompatible.
s3fs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3

### Step 2: Hardware Verification & Dataset Inspection
- Verifies GPU availability (CUDA acceleration is recommended for training).
- Streams the **MUCS-Hinglish** dataset (dianavdavidson/MUCS-Hinglish-traintestblindsplit) without downloading the entire dataset upfront.
- Displays sample transcripts and available audio metadata columns.

In [2]:
import torch
print("GPU:", torch.cuda.is_available())
from datasets import load_dataset
ds = load_dataset("dianavdavidson/MUCS-Hinglish-traintestblindsplit", split="train", streaming=True)
for i, x in zip(range(3), ds):
    print(list(x.keys()), "|", x["transcript"][:80])

GPU: True


README.md:   0%|          | 0.00/1.03k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/23 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/23 [00:00<?, ?it/s]

['audio', 'segment_id', 'transcript', 'speaker_id', 'segment_part', 'hindi_words', 'ratio_english_words', 'ratio_hindi_words', 'english_words', 'count_english_words', 'count_dev_words', 'ratio_english_words_range'] | दोस्तों bash में nested और multilevel if statement के spoken tutorial में आपका स
['audio', 'segment_id', 'transcript', 'speaker_id', 'segment_part', 'hindi_words', 'ratio_english_words', 'ratio_hindi_words', 'english_words', 'count_english_words', 'count_dev_words', 'ratio_english_words_range'] | इस tutorial में हम निम्न के बारे में सीखेंगे
['audio', 'segment_id', 'transcript', 'speaker_id', 'segment_part', 'hindi_words', 'ratio_english_words', 'ratio_hindi_words', 'english_words', 'count_english_words', 'count_dev_words', 'ratio_english_words_range'] | nested ifelse और


### Step 3: Environment Setup & Background Training Launch
- Detects the execution environment (Kaggle or Google Colab) and prepares output and persistent directories.
- Adjusts sample capacity (MAX_SAMPLES_HINGLISH = 20000).
- Spawns the training process (PolyWhisper/train_v3.py) using openai/whisper-small with encoder + decoder LoRA.
- Periodically mirrors checkpoint progress and outputs training logs.

In [3]:
import os, sys, shutil, subprocess, time, glob

EPOCHS, MAX_HOURS, N_SAMPLES = 3, 8, 20000

ON_KAGGLE = os.path.exists("/kaggle/working")
ON_COLAB = "google.colab" in sys.modules
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
OUT = f"{WORK}/hinglish_out"
PERSIST = None
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST = "/content/drive/MyDrive/mediscribe_hinglish"
os.makedirs(f"{OUT}/adapters_v3", exist_ok=True)

KEEP = ["adapters_v3/*.pt", "training_state_v3_hing.json", "training_v3.log"]
def mirror(src, dst):
    for pat in KEEP:
        for f in glob.glob(f"{src}/{pat}"):
            d = f.replace(src, dst, 1)
            os.makedirs(os.path.dirname(d), exist_ok=True)
            shutil.copy2(f, d)

if PERSIST and os.path.exists(PERSIST):
    mirror(PERSIST, OUT); print("Restored previous run from Drive")

# the script hard-codes the Hinglish cap at 42000; lower it
!sed -i 's/^MAX_SAMPLES_HINGLISH = .*/MAX_SAMPLES_HINGLISH = {N_SAMPLES}/' PolyWhisper/train_v3.py
!grep -n "^MAX_SAMPLES_HINGLISH" PolyWhisper/train_v3.py

cmd = [sys.executable, "train_v3.py", "--langs", "hinglish", "--model-size", "small", "--encoder-lora",
       "--epochs", str(EPOCHS), "--batch-size", "4", "--tag", "_hing", "--save-dir", OUT,
       "--max-runtime-hours", str(MAX_HOURS), "--no-spec-augment", "--no-speed-perturb"]
proc = subprocess.Popen(cmd, cwd="PolyWhisper", stdout=open(f"{OUT}/run.log", "a"), stderr=subprocess.STDOUT)

while proc.poll() is None:
    time.sleep(300)
    if PERSIST: mirror(OUT, PERSIST)
    print(subprocess.run(["tail", "-n", "3", f"{OUT}/training_v3.log"], capture_output=True, text=True).stdout)
if PERSIST: mirror(OUT, PERSIST)
print("exit code:", proc.returncode)
if proc.returncode: print(open(f"{OUT}/run.log").read()[-3000:])

39:MAX_SAMPLES_HINGLISH = 20000
[2026-10-15 13:59:19]   Freed datasets cache: /root/.cache/huggingface/datasets
[2026-10-15 13:59:19]   Saved 19965 samples
[2026-10-15 13:59:19]   Downloading mucs_hinglish hinglish (test)...

[2026-10-15 14:04:18]   [420/4992] loss=1.2146 lr=9.99e-05 1.6 steps/s ETA=46m mem=0.0GB
[2026-10-15 14:04:21]   [425/4992] loss=1.3020 lr=9.99e-05 1.6 steps/s ETA=46m mem=0.0GB
[2026-10-15 14:04:24]   [430/4992] loss=1.1971 lr=9.99e-05 1.6 steps/s ETA=46m mem=0.0GB

[2026-10-15 14:09:17]   [915/4992] loss=1.5934 lr=9.93e-05 1.7 steps/s ETA=41m mem=0.0GB
[2026-10-15 14:09:20]   [920/4992] loss=0.9683 lr=9.93e-05 1.7 steps/s ETA=41m mem=0.0GB
[2026-10-15 14:09:23]   [925/4992] loss=0.9774 lr=9.92e-05 1.7 steps/s ETA=41m mem=0.0GB

[2026-10-15 14:14:17]   [1410/4992] loss=0.9824 lr=9.81e-05 1.7 steps/s ETA=36m mem=0.0GB
[2026-10-15 14:14:20]   [1415/4992] loss=0.7605 lr=9.81e-05 1.7 steps/s ETA=36m mem=0.0GB
[2026-10-15 14:14:23]   [1420/4992] loss=0.8140 lr=9.81e-0

### Step 4: Model Evaluation & Word Error Rate (WER) Benchmark
- Dynamically loads the PolyWhisperV3 architecture and attaches the best trained LoRA adapter weights (hinglish_best_hing.pt).
- Loads the vanilla openai/whisper-small baseline for side-by-side comparison.
- Evaluates both models on held-out test audio samples and measures Word Error Rate (WER) using jiwer.

In [4]:
import json, re, unicodedata, glob, soundfile as sf, jiwer, torch, torch.nn as nn
from transformers import WhisperProcessor, WhisperForConditionalGeneration

dev = "cuda" if torch.cuda.is_available() else "cpu"
# load only the model class from the repo's script (avoids executing the training code)
src = open("PolyWhisper/train_v3.py").read()
ns = dict(torch=torch, nn=nn, WhisperForConditionalGeneration=WhisperForConditionalGeneration,
          WHISPER_MODEL="openai/whisper-small", RANK=16, ENCODER_LORA=True, BOS_EOS=50257)
exec(src[src.index("class PolyWhisperV3"):src.index("def quick_wer")], ns)

ad = (glob.glob(f"{OUT}/adapters_v3/hinglish_best_wer_hing.pt") or glob.glob(f"{OUT}/adapters_v3/hinglish_best_hing.pt"))[0]
print("Using adapter:", ad)
ft = ns["PolyWhisperV3"]().to(dev).eval()
ft.load_adapter("hinglish", ad)
base = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to(dev).eval()
pw = WhisperProcessor.from_pretrained("openai/whisper-small")

rows = json.load(open(f"{OUT}/data/mucs_hinglish_hinglish_test.json"))[:200]
def norm(t):
    t = unicodedata.normalize("NFC", t.lower())
    return " ".join(re.sub(r"[^\w\s]|_", " ", t).split())

def run(gen):
    hyps = []
    for r in rows:
        a, _ = sf.read(r["wav"])
        f = pw.feature_extractor([a], sampling_rate=16000, return_tensors="pt").input_features.to(dev)
        with torch.no_grad():
            hyps.append(pw.decode(gen(f)[0], skip_special_tokens=True).strip())
    return hyps

h_base = run(lambda f: base.generate(f, language="hi", task="transcribe", max_new_tokens=256, num_beams=1))
h_ft = run(lambda f: ft.generate(f, lang="hinglish", language="hi", task="transcribe", max_new_tokens=256, num_beams=1))
refs = [norm(r["text"]) for r in rows]
print("Baseline whisper-small WER:", round(jiwer.wer(refs, [norm(h) for h in h_base]), 3))
print("Fine-tuned (LoRA) WER:     ", round(jiwer.wer(refs, [norm(h) for h in h_ft]), 3))
for i in range(3): print("\nREF:", rows[i]["text"], "\nBASE:", h_base[i], "\nFT:  ", h_ft[i])

Using adapter: /kaggle/working/hinglish_out/adapters_v3/hinglish_best_hing.pt


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_ne

Baseline whisper-small WER: 2.407
Fine-tuned (LoRA) WER:      0.828

REF: लिबर ऑफिस impress में एक प्रस्तुति document बनाना और बुनियादी formatting के इस spoken tutorial में आपका स्वागत है 
BASE: लिबर अफिस इंप्रस में एक प्रस्तुती डोक्यमें बनाना और बून्यादी फार्मेटिं के इस पोगें तीटौल में अपका स्वागात है. 
FT:   अब यहाँ इसके बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे बारे

REF: इस tutorial में हम impress window के भागों के बारे में सीखेंगे और कैसे स्लाइड इन्सर्ट करें और कॉपी करें फॉन्ट तथा फॉन्ट को फॉर्मेट करना सीखेंगे 
BASE: अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अप 
FT:   �इस tutorial में हम impress window के भागों के बारे में सी

### Step 5: Export Adapter Checkpoint
- Copies the best-performing adapter weights to hinglish_adapter.pt for deployment and integration into MediScribe.

In [5]:
shutil.copy(ad, f"{WORK}/hinglish_adapter.pt"); print("Saved", f"{WORK}/hinglish_adapter.pt")

Saved /kaggle/working/hinglish_adapter.pt
